#### Objectif

Ce notebook sert de workflow d'orchestration pour la partie exploratoire Morlet. 
Toute la logique de lecture, prétraitement, epoching, calcul temps-fréquence, normalisation baseline et génération de figures est centralisée dans lfp_utils.py.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from lfp_preprocess import (
    list_trc_sessions,
    load_bad_channels_table
)

from lfp_morlet_utils import (
    MorletConfig,
    prepare_session_data,
    run_session_morlet,
    run_all_sessions_morlet,
    load_session_exports,
    generate_session_figures_from_exports
)

In [2]:
# configuration Morlet

morlet_cfg = MorletConfig(
    root_dir="/home/aube/Documents/article_neuronal_stimic/effets_cog/theta_gamma_cog",
    output_dir="/home/aube/Documents/article_neuronal_stimic/effets_cog/theta_gamma_cog/results_morlet_exploratoire",

    # Epoching
    pre_length=3.0,
    post_length=3.0,
    epsilon=0.1,
    allow_global_baseline=True,

    # Filtrage
    do_notch=True,
    notch_freqs=(50.0, 100.0, 150.0),
    notch_q=30.0,
    do_highpass=True,
    highpass_hz=0.1,

    # Morlet
    fmin=4.0,
    fmax=150.0,
    n_freqs=20,
    freq_scale="linear",
    n_cycles=7.0,
    decim=16,
    n_jobs=1,

    # Baseline
    baseline_mode="trial_pre",          # recommandé pour la phase exploratoire
    baseline_stat="median",
    metrics_to_compute=("logratio", "percent", "zscore", "subtract"),
    eps=1e-12,

    # Figures
    make_figures=True,
    figure_dpi=150,
    max_cols_per_figure=3,
    cmap_raw="viridis",
    cmap_metric_div="RdBu_r",
    raw_display_mode="log10",
    z_threshold=3.0,
    lineplot_time_stat="mean",
    save_png=True,

    # Logs
    verbose=True,
)

In [3]:
# quelles sessions dispo ?
root_dir = Path(morlet_cfg.root_dir)
sessions = list_trc_sessions(root_dir)
print(f"{len(sessions)} sessions trouvées")
sessions

2 sessions trouvées


['P101_DC54_stim2', 'P64_BR34_stim2']

In [4]:
# quels bad channels ?
bad_df = load_bad_channels_table(root_dir)
bad_df

,session,bad_channels
0,P64_BR34_stim2,"B14,C15,GC6,GC7,W46"
1,P101_DC54_stim2,"B_1,H_11,CU_12,Bp8"


#### Morlet sur une session

In [5]:
# pour test sur session unique
session = sessions[1]   # ou nom spécifique de session
session

'P64_BR34_stim2'

In [ ]:
# preparation de la session (a mettre dans boucle sur sessions apres)

session_data = prepare_session_data(
    session=session,
    root_dir=root_dir,
    bad_df=bad_df,
    cfg=morlet_cfg,
)

print("Clés disponibles :", session_data.keys())
print("Session :", session_data["session"])
print("sfreq :", session_data["sfreq"])
print("n_stims :", len(session_data["stims_df"]))
print("n_raw_channels :", len(session_data["raw_ch_names"]))
print("n_bad_channels :", len(session_data["bad_channels"]))
print("n_bipolar_channels :", len(session_data["bp_names"]))
print("pre_epochs shape :", session_data["pre_epochs"].shape)
print("post_epochs shape :", session_data["post_epochs"].shape)

session_data["trials_df"]

In [7]:
# Morlet sur cette session 
session_out = run_session_morlet(
    session_data=session_data,
    out_dir=Path(morlet_cfg.output_dir),
    cfg=morlet_cfg,
)

session_out


=== Morlet session P64_BR34_stim2 ===
[INFO] P64_BR34_stim2: Morlet canal 1/110 -> A1-A2
[INFO] P64_BR34_stim2 A1-A2: pre_ep_ch shape = (15, 1, 6144)
[INFO] P64_BR34_stim2 A1-A2: post_ep_ch shape = (15, 1, 6144)
[INFO] P64_BR34_stim2 A1-A2: power_pre shape = (15, 1, 20, 384)
[INFO] P64_BR34_stim2 A1-A2: power_post shape = (15, 1, 20, 384)
[INFO] P64_BR34_stim2 A1-A2: sauvegarde canal OK
[INFO] P64_BR34_stim2: Morlet canal 2/110 -> A2-A3
[INFO] P64_BR34_stim2 A2-A3: pre_ep_ch shape = (15, 1, 6144)
[INFO] P64_BR34_stim2 A2-A3: post_ep_ch shape = (15, 1, 6144)
[INFO] P64_BR34_stim2 A2-A3: power_pre shape = (15, 1, 20, 384)
[INFO] P64_BR34_stim2 A2-A3: power_post shape = (15, 1, 20, 384)
[INFO] P64_BR34_stim2 A2-A3: sauvegarde canal OK
[INFO] P64_BR34_stim2: Morlet canal 3/110 -> A3-A4
[INFO] P64_BR34_stim2 A3-A4: pre_ep_ch shape = (15, 1, 6144)
[INFO] P64_BR34_stim2 A3-A4: post_ep_ch shape = (15, 1, 6144)
[INFO] P64_BR34_stim2 A3-A4: power_pre shape = (15, 1, 20, 384)
[INFO] P64_BR34_sti

PosixPath('/home/aube/Documents/article_neuronal_stimic/effets_cog/results_morlet_exploratoire/P64_BR34_stim2')

In [9]:
morlet_cfg.output_dir + '/' + session

'/home/aube/Documents/article_neuronal_stimic/effets_cog//results_morlet_exploratoire/P64_BR34_stim2'

In [11]:
# reload exports de Morlet pour verification
session_out = Path(morlet_cfg.output_dir + '/' + session)
exports = load_session_exports(session_out, session)

print("Session exportée :", exports["session"])
print("freqs shape :", exports["freqs"].shape)
print("post_times shape :", exports["post_times"].shape)
print("n_trials exportés :", len(exports["stims_df"]))
print("n_bipolar_channels exportés :", len(exports["metadata"]["bipolar_names"]))

Session exportée : P64_BR34_stim2
freqs shape : (20,)
post_times shape : (384,)
n_trials exportés : 15
n_bipolar_channels exportés : 110


In [12]:
# regenerate figures from exports (from .npy)
session_out = Path(morlet_cfg.output_dir + '/' + session)
generate_session_figures_from_exports(
    session_dir=session_out,
    cfg=morlet_cfg,
)

[OK] P64_BR34_stim2: figures exploratoires générées


#### Morlet sur toutes les sessions

In [17]:
# Morlet sur toutes les sessions
summary_morlet = run_all_sessions_morlet(morlet_cfg)
summary_morlet

2 sessions TRC trouvées

=== Préparation session P101_DC54_stim2 ===


[INFO] P101_DC54_stim2.TRC: sfreq = 2048.0
[INFO] P101_DC54_stim2.TRC: unités canal détectées = ['%', 'bpm', 'dimentionless', 'μV']

=== Morlet session P101_DC54_stim2 ===
[INFO] P101_DC54_stim2: Morlet canal 1/89 -> A1-A2
[INFO] P101_DC54_stim2 A1-A2: pre_ep_ch shape = (42, 1, 6144)
[INFO] P101_DC54_stim2 A1-A2: post_ep_ch shape = (42, 1, 6144)
[INFO] P101_DC54_stim2 A1-A2: power_pre shape = (42, 1, 20, 384)
[INFO] P101_DC54_stim2 A1-A2: power_post shape = (42, 1, 20, 384)
[INFO] P101_DC54_stim2 A1-A2: sauvegarde canal OK
[INFO] P101_DC54_stim2: Morlet canal 2/89 -> A2-A3
[INFO] P101_DC54_stim2 A2-A3: pre_ep_ch shape = (42, 1, 6144)
[INFO] P101_DC54_stim2 A2-A3: post_ep_ch shape = (42, 1, 6144)
[INFO] P101_DC54_stim2 A2-A3: power_pre shape = (42, 1, 20, 384)
[INFO] P101_DC54_stim2 A2-A3: power_post shape = (42, 1, 20, 384)
[INFO] P101_DC54_stim2 A2-A3: sauvegarde canal OK
[INFO] P101_DC54_stim2: Morlet canal 3/89 -> A6-A7
[INFO] P101_DC54_stim2 A6-A7: pre_ep_ch shape = (42, 1, 6144)
[

[INFO] P64_BR34_stim2.TRC: sfreq = 2048.0
[INFO] P64_BR34_stim2.TRC: unités canal détectées = ['%', 'bpm', 'dimentionless', 'μV']

=== Morlet session P64_BR34_stim2 ===
[INFO] P64_BR34_stim2: Morlet canal 1/110 -> A1-A2
[INFO] P64_BR34_stim2 A1-A2: pre_ep_ch shape = (15, 1, 6144)
[INFO] P64_BR34_stim2 A1-A2: post_ep_ch shape = (15, 1, 6144)
[INFO] P64_BR34_stim2 A1-A2: power_pre shape = (15, 1, 20, 384)
[INFO] P64_BR34_stim2 A1-A2: power_post shape = (15, 1, 20, 384)
[INFO] P64_BR34_stim2 A1-A2: sauvegarde canal OK
[INFO] P64_BR34_stim2: Morlet canal 2/110 -> A2-A3
[INFO] P64_BR34_stim2 A2-A3: pre_ep_ch shape = (15, 1, 6144)
[INFO] P64_BR34_stim2 A2-A3: post_ep_ch shape = (15, 1, 6144)
[INFO] P64_BR34_stim2 A2-A3: power_pre shape = (15, 1, 20, 384)
[INFO] P64_BR34_stim2 A2-A3: power_post shape = (15, 1, 20, 384)
[INFO] P64_BR34_stim2 A2-A3: sauvegarde canal OK
[INFO] P64_BR34_stim2: Morlet canal 3/110 -> A3-A4
[INFO] P64_BR34_stim2 A3-A4: pre_ep_ch shape = (15, 1, 6144)
[INFO] P64_BR34

{'n_sessions': 2,
 'n_errors': 0,
 'errors': [],
 'config': {'root_dir': '/home/aube/Documents/article_neuronal_stimic/effets_cog/theta_gamma_cog',
  'output_dir': '/home/aube/Documents/article_neuronal_stimic/effets_cog//results_morlet_exploratoire',
  'pre_length': 3.0,
  'post_length': 3.0,
  'epsilon': 0.1,
  'allow_global_baseline': True,
  'do_notch': True,
  'notch_freqs': (50.0, 100.0, 150.0),
  'notch_q': 30.0,
  'do_highpass': True,
  'highpass_hz': 0.1,
  'fmin': 4.0,
  'fmax': 150.0,
  'n_freqs': 20,
  'freq_scale': 'linear',
  'n_cycles': 7.0,
  'decim': 16,
  'n_jobs': 1,
  'baseline_mode': 'trial_pre',
  'baseline_stat': 'median',
  'metrics_to_compute': ('logratio', 'percent', 'zscore', 'subtract'),
  'eps': 1e-12,
  'save_raw_epochs': False,
  'save_filtered_signal_preview': False,
  'make_figures': True,
  'figure_dpi': 150,
  'max_cols_per_figure': 3,
  'cmap_raw': 'viridis',
  'cmap_metric_div': 'RdBu_r',
  'raw_display_mode': 'log10',
  'z_threshold': 3.0,
  'linep

In [18]:
# résumé global d'execution
summary_file = Path(morlet_cfg.output_dir) / "run_summary.json"
pd.read_json(summary_file)

ValueError: Mixing dicts with non-Series may lead to ambiguous ordering.

In [ ]:
# Variante de Morlet sans figures, et sans verbose, pour accélérer le calcul et creation des .npy
morlet_cfg_fast = MorletConfig(
    **{**morlet_cfg.__dict__, "make_figures": False, "verbose": False}
)

In [ ]:
# Variante de Morlet avec baseline globale au lieu de baseline pre-stim
morlet_cfg_global_base = MorletConfig(
    **{
        **morlet_cfg.__dict__,
        "baseline_mode": "global_pre_first_stim",
        "allow_global_baseline": True,
    }
)

In [ ]:
# Variante de Morlet avec exploration fréquentielle plus importante
morlet_cfg_dense = MorletConfig(
    **{
        **morlet_cfg.__dict__,
        "n_freqs": 40,
        "freq_scale": "semilog",
    }
)